# Intensity, sampling and forecasting

A Hawkes process remembers its past: each event raises the intensity, which then decays. This
notebook takes one observed history and asks what happens next:

1. the intensity on the history,
2. many sampled futures (multi-step forecasts) from that history,
3. what changes when the observation continued, event-free, after the last event (`T_obs`).

The parameters are set by hand so the behaviour is easy to read. For the step-by-step definitions see
[`00_tpp_introduction.ipynb`](./00_tpp_introduction.ipynb). Runs on CPU in a few seconds.

In [ ]:
import matplotlib.pyplot as plt
import torch

import torchpp as tpp
from torchpp.modules.intensity import ConstantBaselineParams, ExpKernelParams

torch.set_default_dtype(torch.float64)

model = tpp.ConstBaselineExpKernelHawkesProcess(
    D=2,
    baseline_params=ConstantBaselineParams(mu=torch.tensor([0.2, 0.1])),
    kernel_params=ExpKernelParams(alpha=torch.tensor([[0.5, 0.2], [0.3, 0.4]]),
                                  beta=torch.tensor([[2.0, 1.0], [1.0, 1.5]])),
)
history = tpp.MVEventData(torch.tensor([0.3, 0.9, 1.1, 1.6, 2.8, 3.0]), torch.tensor([0, 0, 1, 0, 1, 1]))
COLORS = ["tab:blue", "tab:red"]

## The intensity on the history

`intensity(t, batch)` takes query times of shape `(B, Q)` and returns `(B, Q, D)`.

In [ ]:
batch = tpp.BatchedMVEventData(mv_events=[history])
grid = torch.linspace(0, 4, 800)
with torch.no_grad():
    lam = model.intensity(grid.unsqueeze(0), batch)[0]

fig, ax = plt.subplots(figsize=(9, 3))
for d in range(2):
    ax.plot(grid, lam[:, d], color=COLORS[d], label=f"type {d}")
    ev = history.time_points[history.event_types == d]
    ax.plot(ev, torch.zeros_like(ev), "|", color=COLORS[d], markersize=14)
ax.set_xlabel("time"); ax.set_ylabel("$\\lambda_d(t)$"); ax.legend(); plt.show()

## Many futures from one history

`sample(history, num_steps=k)` draws `k` events one after the other; each new event becomes part of
the history for the next one. Repeating it with different seeds gives a set of possible futures.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
for d in range(2):
    ev = history.time_points[history.event_types == d]
    ax.plot(ev, [-1] * len(ev), "o", color=COLORS[d])
with torch.no_grad():
    for k in range(15):
        future = model.sample(history, num_steps=8, rng=k).sequence
        new_t = future.time_points[len(history):]
        new_m = future.event_types[len(history):]
        ax.scatter(new_t, [k] * len(new_t), c=[COLORS[m] for m in new_m.tolist()], s=12)
ax.axvline(float(history.time_points[-1]), color="k", lw=0.8)
ax.set_yticks([-1, 7], ["history", "sampled futures"]); ax.set_xlabel("time"); ax.set_xlim(0, 14)
plt.show()

Right after the history the futures are dense, because the recent events still excite the process;
further out they thin to the baseline rate. Summaries of the next event come without sampling:

In [ ]:
with torch.no_grad():
    print("mean time of the next event:", round(model.time_mean(history).item(), 3))
    print("median:", round(model.time_quantile(0.5, history).item(), 3))
    print("p(type) of the next event:", model.prob_mark(history, log=False)[0].numpy().round(3))

## Forecasting from the end of the observation

Suppose we kept watching until `T_obs = 5` and saw nothing after the last event at 3.0. Then the next
event is after 5, and the quiet stretch is evidence that the excitation has faded. With `T_obs` set,
every prediction conditions on "no event in (3.0, 5]".

In [ ]:
watched = tpp.MVEventData(history.time_points, history.event_types, T_obs=5.0)
t = torch.linspace(5.0, 12.0, 300).unsqueeze(0)
t_full = torch.linspace(3.0 + 1e-4, 12.0, 400).unsqueeze(0)  # just after the last event
with torch.no_grad():
    p_last = model.density_time(t_full, history, log=False)[0]
    p_obs = model.density_time(t, watched, log=False)[0]
    print("mean next-event time, from the last event :", round(model.time_mean(history).item(), 3))
    print("mean next-event time, given nothing until 5:", round(model.time_mean(watched).item(), 3))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t_full[0], p_last, label="from the last event (T_obs = 3.0)")
ax.plot(t[0], p_obs, label="given no event until T_obs = 5")
ax.axvline(5.0, color="k", lw=0.6, ls="--")
ax.set_xlabel("time"); ax.set_ylabel("$p(t)$"); ax.legend(); plt.show()

The conditioned density is the original one restricted to $t > 5$ and rescaled:
$p(t \mid \text{no event until } 5) = p(t) / S(5)$. Samples drawn from `watched` start after 5 as well.

Next: write your own model in [`04_add_a_model.ipynb`](./04_add_a_model.ipynb); to fit and compare
models, see [`06_compare_models.ipynb`](./06_compare_models.ipynb).